In [ ]:
using QuantumOptics
using OrdinaryDiffEq
using LinearAlgebra
using Plots

# Parameters
Lx = 5
Ly = 5             # Lattice size (5x5)
N_sites = Lx * Ly
wc = 1.0            # Cavity Frequency
wa = 1.0            # Atom Frequency
g  = 0.05           # Coupling Strength
J  = 0.1            # Hopping Strength
dim_ph = 5          # Photon cutoff (0, 1, 2)

# Define the basis for one site (cavity + atom)
b_cav = FockBasis(dim_ph - 1)
b_atom = SpinBasis(1//2)
b_site = b_cav ⊗ b_atom 
d_site = length(b_site) # Dimension of one site (e.g. 3*2 = 6)

# Operators for ONE site
a  = destroy(b_cav) ⊗ one(b_atom)
at = create(b_cav) ⊗ one(b_atom)
sz = one(b_cav) ⊗ sigmaz(b_atom)
sm = one(b_cav) ⊗ sigmam(b_atom)
sp = one(b_cav) ⊗ sigmap(b_atom)
n_op = number(b_cav) ⊗ one(b_atom)

# Local Hamiltonian (Internal energy of one site)
H_local = wc * n_op + 0.5 * wa * sz + g * (at * sm + a * sp)

# Define Lattice and Neighbors 
neighbors = [Int[] for _ in 1:N_sites]
for x in 1:Lx
    for y in 1:Ly
        i = x + (y-1)*Lx 
        if x < Lx; right = (x+1) + (y-1)*Lx; push!(neighbors[i], right); push!(neighbors[right], i); end
        if y < Ly; up = x + ((y+1)-1)*Lx; push!(neighbors[i], up); push!(neighbors[up], i); end
    end
end

# The Mean Field Function to compute d|psi>/dt
function mean_field_update!(dpsi_flat, psi_flat, p, t)
    # We create a temporary list of Kets for calculation
    psi_kets = [Ket(b_site, psi_flat[(i-1)*d_site+1 : i*d_site]) for i in 1:N_sites]
    
    # Calculate Field Expectations <a_j>
    alphas = [expect(a, k) for k in psi_kets]
    
    # Update every site
    for i in 1:N_sites
        # Sum neighbor fields
        field_sum = sum(alphas[neigh] for neigh in neighbors[i])
        
        # Effective Hamiltonian
        H_eff = H_local - J * (at * field_sum + a * conj(field_sum))
        
        # Calculate update: d|psi> = -i * H * |psi>
        dpsi_ket = -1.0im * (H_eff * psi_kets[i])
        
        # Write back to flat derivative array
        dpsi_flat[(i-1)*d_site+1 : i*d_site] = dpsi_ket.data
    end
end

# Initial State
# We use a Coherent State instead of a Fock state. 
# A Coherent state has < a > != 0, so neighbors can "feel" it.
alpha = 1.0 # Average photon number approx |alpha|^2 = 1

# Create initial Kets
psi0_kets = [fockstate(b_cav, 0) ⊗ spindown(b_atom) for _ in 1:N_sites]
center = Int(ceil(N_sites/2))

# Use coherentstate here:
psi0_kets[center] = coherentstate(b_cav, alpha) ⊗ spindown(b_atom)

# Flatten into one big vector
u0 = reduce(vcat, [k.data for k in psi0_kets])

# Run Simulation
tspan = (0.0, 20.0)
# Pass the FLAT vector u0
prob = ODEProblem(mean_field_update!, u0, tspan)
sol = solve(prob, Tsit5(), saveat=0.1)

# Analyze Results
times = sol.t
n_t = zeros(length(times), N_sites)

for (t_idx, u_flat) in enumerate(sol.u)
    # Reconstruct Kets from the flat solution to measure observables
    for i in 1:N_sites
        # Extract the chunk for site i
        psi_chunk = u_flat[(i-1)*d_site+1 : i*d_site]
        k = Ket(b_site, psi_chunk)
        
        # Measure photon number
        n_t[t_idx, i] = real(expect(n_op, k))
    end
end

# Plotting
plot(times, n_t[:, center], label="Center Site", xlabel="Time", ylabel="Photon Number", lw=2)
plot!(times, n_t[:, center+1], label="Neighbor Site", title="Mean Field Dynamics", lw=2)

In [ ]:
using DelimitedFiles

# Combine Time, Center Site Density, and Neighbor Density into one matrix
# Columns: [Time, n_center, n_neighbor]
data_to_save = hcat(times, n_t[:, center], n_t[:, center+1])

# Save to a text file
writedlm("meanfield_5x5_results.txt", data_to_save)
println("Data saved to meanfield_5x5_results.txt")

In [ ]:
using Plots

# Animation Code
# Reshape the flat data back into 5x5 grids for plotting
anim = @animate for t_idx in 1:length(times)
    # Extract density for all sites at time t
    densities = n_t[t_idx, :]
    
    # Reshape into 5x5 matrix
    grid_data = reshape(densities, (Lx, Ly))
    
    # Plot heatmap
    heatmap(grid_data, 
            clims=(0, 0.5), # Set max to 0.5 to see contrast better
            title="Photon Distribution t=$(round(times[t_idx], digits=1))", 
            aspect_ratio=:equal,
            c=:viridis)
end

gif(anim, "jch_5x5_spread.gif", fps=15)
println("Animation saved as jch_5x5_spread.gif")